In [ ]:
######################
## K-NN CLASSIFIERS ##
######################

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

"""
# Filter out entries where ggrade is not 1-4
valid_mask = np.isin(ggrade, [1, 2, 3, 4])
X = np.array(reduced_embeddings)[valid_mask]
y_id = np.array(numeric_ids)[valid_mask]
y_grade = np.array(ggrade)[valid_mask]

"""

# Convert data to numpy arrays ()
X = np.array(reduced_embeddings)
y_id = np.array(numeric_ids)
y_grade = np.array(ggrade)

# First split: train vs temp (60/40)
X_train, X_temp, y_id_train, y_id_temp, y_grade_train, y_grade_temp = train_test_split(
    X, y_id, y_grade, test_size=0.4, random_state=42)

# Second split: val vs test (20/20)
X_val, X_test, y_id_val, y_id_test, y_grade_val, y_grade_test = train_test_split(
    X_temp, y_id_temp, y_grade_temp, test_size=0.5, random_state=42)

best_k_id = 1
best_acc_id = 0
best_knn_id = None

best_k_grade = 1
best_acc_grade = 0
best_knn_grade = None

for k in range(1, 21):  # Try k from 1 to 20
    # numeric_ids
    knn_id = KNeighborsClassifier(n_neighbors=k)
    knn_id.fit(X_train, y_id_train)
    acc_id = knn_id.score(X_val, y_id_val)
    if acc_id > best_acc_id:
        best_k_id = k
        best_acc_id = acc_id
        best_knn_id = knn_id
    
    # ggrade
    knn_grade = KNeighborsClassifier(n_neighbors=k)
    knn_grade.fit(X_train, y_grade_train)
    acc_grade = knn_grade.score(X_val, y_grade_val)
    if acc_grade > best_acc_grade:
        best_k_grade = k
        best_acc_grade = acc_grade
        best_knn_grade = knn_grade

print(f"Best k for numeric_ids: {best_k_id}, Validation Accuracy: {best_acc_id:.2f}")
print(f"Best k for ggrade: {best_k_grade}, Validation Accuracy: {best_acc_grade:.2f}")

# Train k-NN classifier for numeric_ids
knn_id = KNeighborsClassifier(n_neighbors=3)
knn_id.fit(X_train, y_id_train)
y_id_pred = knn_id.predict(X_test)
print("ID Accuracy:", accuracy_score(y_id_test, y_id_pred))

# Train k-NN classifier for ggrade
knn_grade = KNeighborsClassifier(n_neighbors=3)
knn_grade.fit(X_train, y_grade_train)
y_grade_pred = knn_grade.predict(X_test)
print("Grade Accuracy:", accuracy_score(y_grade_test, y_grade_pred))

#
###

In [ ]:
##########################################
## VISUALISATIONS OF CONFUSION MATRICES ##
##########################################

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_confusion_matrices(knn, X_sets, y_sets, label_name):
    split_names = ['Train', 'Validation', 'Test']
    
    for X, y_true, name in zip(X_sets, y_sets, split_names):
        y_pred = knn.predict(X)
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(cmap='Blues', values_format='d')
        plt.title(f'{label_name} Confusion Matrix ({name} Set)')
        plt.show()

# Re-train models (just in case)
knn_id = KNeighborsClassifier(n_neighbors=3)
knn_id.fit(X_train, y_id_train)

knn_grade = KNeighborsClassifier(n_neighbors=3)
knn_grade.fit(X_train, y_grade_train)

# Prepare data splits
X_sets = [X_train, X_val, X_test]
y_id_sets = [y_id_train, y_id_val, y_id_test]
y_grade_sets = [y_grade_train, y_grade_val, y_grade_test]

# Plot confusion matrices
plot_confusion_matrices(knn_id, X_sets, y_id_sets, "Numeric ID")
plot_confusion_matrices(knn_grade, X_sets, y_grade_sets, "Cancer Grade")

#
###